# Import Libraries

In [1]:
import sys
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# Load Dataset

In [3]:
sys.path.append(str(Path().resolve().parents[1]))
from dm_classif.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR

import pandas as pd

# Load the CSV
Interim_data_path = INTERIM_DATA_DIR / "diabetes_data_interim.csv"
df = pd.read_csv(Interim_data_path)

In [4]:
df.shape

(69057, 22)

# Feature Selection

In [5]:
# Define features and target
X = df.drop('Diabetes_binary', axis=1) 
y = df['Diabetes_binary']

# Feature selection using SelectKBest with Chi-Square
selector = SelectKBest(score_func=mutual_info_classif, k=12)
X_new = selector.fit_transform(X, y)

# Get the selected feature indices
selected_columns = selector.get_support(indices=True)
important_features = X.columns[selected_columns].tolist()

# Display the selected features
print(important_features)

['HighBP', 'HighChol', 'CholCheck', 'BMI', 'HeartDiseaseorAttack', 'PhysActivity', 'GenHlth', 'PhysHlth', 'DiffWalk', 'Age', 'Education', 'Income']


### Creating new DataFrame form selected features

In [6]:
X_selected = pd.DataFrame(X_new, columns=important_features, index=X.index)

### Splitting dataset

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)

In [8]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(55245, 12) (13812, 12) (55245,) (13812,)


### Scaling

In [9]:
binary_cols = [col for col in important_features if df[col].nunique() == 2]
scale_cols = [col for col in important_features if col not in binary_cols]

In [10]:
scaler = RobustScaler()

# Scale only non-binary (continuous) columns
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

In [11]:
train_processed = pd.concat([X_train_scaled, y_train], axis=1)
test_processed = pd.concat([X_test_scaled, y_test], axis=1)

In [12]:
train_processed.columns

Index(['HighBP', 'HighChol', 'CholCheck', 'BMI', 'HeartDiseaseorAttack',
       'PhysActivity', 'GenHlth', 'PhysHlth', 'DiffWalk', 'Age', 'Education',
       'Income', 'Diabetes_binary'],
      dtype='object')

In [13]:
train_processed.head()

,HighBP,HighChol,CholCheck,BMI,HeartDiseaseorAttack,PhysActivity,GenHlth,PhysHlth,DiffWalk,Age,Education,Income,Diabetes_binary
11041,0.0,1.0,1.0,-0.625,0.0,1.0,-0.5,0.000000,0.0,0.25,0.0,0.25,0.0
27964,1.0,0.0,1.0,-0.375,0.0,1.0,-0.5,1.428571,1.0,1.00,0.5,0.25,0.0
30967,1.0,1.0,1.0,-0.250,0.0,1.0,0.0,0.000000,0.0,-0.25,-0.5,-0.75,0.0
895,0.0,1.0,1.0,-0.875,0.0,1.0,-1.0,0.000000,0.0,0.00,0.0,0.50,0.0
67503,1.0,0.0,1.0,1.125,0.0,1.0,0.0,0.000000,0.0,0.75,-0.5,-0.50,1.0


In [14]:
# Save to data/processed
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
train_path = PROCESSED_DATA_DIR / "train_processed.csv"
test_path = PROCESSED_DATA_DIR / "test_processed.csv"

train_processed.to_csv(train_path, index=False)
test_processed.to_csv(test_path, index=False)

print(f"Saved: {train_path.name}, {test_path.name}")

Saved: train_processed.csv, test_processed.csv
